# Phase 2 — Same-Cell Tracking Across Sessions (DREADD saline/DCZ cohort)

Goal: track individual cells across a **caller-chosen small group of sessions**
(e.g. BaselineDay5 + DCZ1_SALINE + DCZ1_DCZ, or an openloop active SALINE/DCZ
pair) rather than across the full ~15-session series per animal — tracking
everything at once would shrink N down to whatever survives every single
session, which isn't realistic. Each comparison group gets its own
registration + master cell-ID table.

Built on top of the existing `TrackROIs.py`/`TrackROIs_SalineDCZ.py`
machinery (FOV alignment via `phase_cross_correlation`, centroid+footprint
candidate matching, Hungarian assignment, auto-accept/reject + manual
review) — generalized to an arbitrary session list + caller-chosen
reference, and joined against Phase 1's per-session layer labels
(`*_layer_curve_results.h5`) to produce the master table.

Built incrementally, one function at a time. Consolidated into `2.CellTracking.py`
only once everything here works end-to-end on real session groups.

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import re
import glob
import json
import numpy as np
import h5py
import pandas as pd
import matplotlib
matplotlib.use('Qt5Agg')  # interactive review popups
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.patches import Circle
from matplotlib.widgets import CheckButtons, Button
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import shift as ndi_shift
from skimage.registration import phase_cross_correlation

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

MICRONS_PER_PIXEL = 1.08952017715202  # V1_prism_DREADD animals (JSY090, JSY093)

# Switched from the disposable TEST_JSY090_V1prism_DREADD copy to the real
# animal directory now that Phase 1 (layer curve assignment) has actually
# been run for real across all 15 recordings here -- Function 2.7 onward
# needs real *_layer_curve_results.h5 files to join against, which only
# exist in this real folder.
TEST_ANIMAL_DIR = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD"

# Defined here (not inside the save cell) so it's available immediately after
# a kernel restart, without needing tracking_result to already exist just to
# save/reload it -- see the checkpoint utility below.
checkpoint_path = os.path.join(TEST_ANIMAL_DIR, '_dev_checkpoints', 'Day1_SalineDCZ1_checkpoint.pkl')


## Function 2.1 — `discover_animal_sessions`

Convenience lookup, not the main driver: scans an animal's whole folder
tree for every `suite2p/plane0` output, and labels each session by
whichever known naming pattern its TSeries/parent folder matches:

- Folder name contains `'skip'` (case-insensitive, anywhere in the TSeries
  or parent folder name) -> **excluded entirely**, not added to the
  catalog. For the case of 2 recordings under the same Day/condition folder
  where you only want one (e.g. Day4's "no water delivered" recording) --
  rename the one to exclude to include "skip" and it's dropped here before
  label matching even happens.
- TSeries folder name contains `SAL` (covers `SALINE`/`SAL`) -> `'saline'`,
  label `'{parent_folder}_SALINE'`
- TSeries folder name contains `DCZ` -> `'dcz'`, label `'{parent_folder}_DCZ'`
- parent folder name matches `Day<N>` (case-insensitive) and neither of the
  above matched -> `'baseline'`, label `'Day<N>'`
- anything else -> `'unknown'`, label = the raw TSeries folder name (printed
  distinctly so nothing gets silently mislabeled -- this is where openloop
  sessions will land until we add their real pattern)

Still guards against **label collisions** as a safety net (e.g. two
non-"skip" recordings that happen to compute the same label) -- the second
one found is skipped with a warning rather than silently overwriting the
first.

- **Input:** `animal_dir` (the animal's top-level folder, e.g.
  `.../JSY090_V1prism_DREADD`).
- **Output:** `catalog`, `{label: {'plane0_path', 'session_type', 'tseries_dir'}}`.

In [ ]:
def discover_animal_sessions(animal_dir):
    """
    Scan animal_dir for every TSeries-*/suite2p/plane0 folder anywhere in
    its tree, labeling each by whichever known naming pattern it matches.
    See markdown above for the exact rules.

    Parameters
    ----------
    animal_dir : str
        Animal's top-level folder.

    Returns
    -------
    catalog : dict
        {label: {'plane0_path': str, 'session_type': str, 'tseries_dir': str}}
    """
    catalog = {}
    unmatched = []
    skipped = []

    plane0_paths = sorted(glob.glob(os.path.join(animal_dir, '**', 'suite2p', 'plane0'),
                                     recursive=True))

    for plane0_path in plane0_paths:
        tseries_dir = os.path.dirname(os.path.dirname(plane0_path))
        tseries_name = os.path.basename(tseries_dir)
        parent_dir = os.path.dirname(tseries_dir)
        parent_name = os.path.basename(parent_dir)

        if 'skip' in tseries_name.lower() or 'skip' in parent_name.lower():
            skipped.append(tseries_dir)
            continue

        upper_tseries = tseries_name.upper()

        if 'SAL' in upper_tseries:
            session_type = 'saline'
            label = f'{parent_name}_SALINE'
        elif 'DCZ' in upper_tseries:
            session_type = 'dcz'
            label = f'{parent_name}_DCZ'
        else:
            day_match = re.search(r'Day(\d+)', parent_name, re.IGNORECASE)
            if day_match:
                session_type = 'baseline'
                label = f'Day{day_match.group(1)}'
            else:
                session_type = 'unknown'
                label = tseries_name

        if label in catalog:
            print(f"WARNING: duplicate label '{label}' -- keeping "
                  f"{catalog[label]['tseries_dir']}, skipping {tseries_dir}")
            continue

        catalog[label] = {
            'plane0_path': plane0_path,
            'session_type': session_type,
            'tseries_dir': tseries_dir,
        }
        if session_type == 'unknown':
            unmatched.append(label)

    print(f"Discovered {len(catalog)} sessions under {animal_dir}:")
    for label, info in catalog.items():
        print(f"  [{info['session_type']:>8}] {label}  <-  {info['tseries_dir']}")

    if skipped:
        print(f"\n{len(skipped)} session(s) excluded via 'skip' in their folder name:")
        for s in skipped:
            print(f"  {s}")

    if unmatched:
        print(f"\n{len(unmatched)} session(s) didn't match a known naming pattern "
              f"(labeled 'unknown', tagged by raw TSeries folder name): {unmatched}")
        print("If any of these are openloop sessions, tell me their real folder "
              "naming convention and I'll add a pattern for them.")

    return catalog


In [ ]:
# --- Try it on the real JSY090 animal directory ---
session_catalog = discover_animal_sessions(TEST_ANIMAL_DIR)


## Function 2.2 — `load_session_for_tracking`

Loads one session's suite2p output (`stat`, `iscell`-filtered) and
reconstructs a dense footprint image + centroid per cell — the same inputs
`TrackROIs`'s matching logic needs (candidate distance from centroids,
correlation from footprints). Directly reimplements
`TrackROIs_SalineDCZ.py`'s `load_suite2p_session` + `reconstruct_footprints`
combined into one call, since we don't edit the original file.

- **Input:** `plane0_path` (from a `discover_animal_sessions` catalog
  entry, or any `.../suite2p/plane0` path).
- **Output:** dict with `stat` (iscell-filtered), `mean_img`, `Ly`, `Lx`,
  `n_cells`, `cell_idx` (iscell-filtered ROI indices — the same convention
  Phase 1's `layer_of_cell` is keyed by), `footprints`
  (`n_cells x Ly x Lx`), `centroids` (`n_cells x 2`, weighted by `lam`).

In [ ]:
def load_session_for_tracking(plane0_path):
    """
    Load one session's suite2p output and reconstruct dense footprints +
    centroids for ROI matching.

    Parameters
    ----------
    plane0_path : str
        Path to a .../suite2p/plane0 folder.

    Returns
    -------
    session_data : dict
        'stat': list of iscell-filtered stat dicts
        'mean_img': (Ly, Lx) array
        'Ly', 'Lx': int
        'n_cells': int
        'cell_idx': (n_cells,) iscell-filtered ROI indices into the raw stat.npy
        'footprints': (n_cells, Ly, Lx) float32, lam-weighted
        'centroids': (n_cells, 2) float64, (y, x), lam-weighted
    """
    plane0_path = str(plane0_path)

    stat = np.load(os.path.join(plane0_path, 'stat.npy'), allow_pickle=True)
    iscell = np.load(os.path.join(plane0_path, 'iscell.npy'), allow_pickle=True)
    ops = np.load(os.path.join(plane0_path, 'ops.npy'), allow_pickle=True).item()

    cell_idx = np.where(iscell[:, 0] == 1)[0]
    stat_cells = [stat[i] for i in cell_idx]

    Ly, Lx = ops['Ly'], ops['Lx']
    n_cells = len(stat_cells)

    footprints = np.zeros((n_cells, Ly, Lx), dtype=np.float32)
    centroids = np.zeros((n_cells, 2), dtype=np.float64)

    for i, s in enumerate(stat_cells):
        ypix, xpix, lam = s['ypix'], s['xpix'], s['lam']
        lam_norm = lam / lam.sum()
        footprints[i, ypix, xpix] = lam
        centroids[i, 0] = np.sum(ypix * lam_norm)
        centroids[i, 1] = np.sum(xpix * lam_norm)

    print(f"  Loaded {plane0_path}: {n_cells} cells, image size {Ly}x{Lx}")

    return {
        'stat': stat_cells,
        'mean_img': ops['meanImg'],
        'Ly': Ly,
        'Lx': Lx,
        'n_cells': n_cells,
        'cell_idx': cell_idx,
        'footprints': footprints,
        'centroids': centroids,
    }


In [ ]:
# --- Choose which sessions to track for THIS comparison, then load just those ---
# Edit SELECTED_LABELS to whichever labels (from session_catalog's printed
# output above) belong to the comparison you want -- e.g. a baseline day +
# one saline/DCZ pair, or an openloop pair. Do NOT load the whole catalog;
# tracking all 15 sessions together is slow and defeats the point (see
# earlier discussion -- small per-comparison groups keep N usable).
SELECTED_LABELS = ['Day1',
                    '260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE',
                    '260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ']

print(f"Loading: {SELECTED_LABELS}\n")

loaded_sessions = {}
for label in SELECTED_LABELS:
    loaded_sessions[label] = load_session_for_tracking(session_catalog[label]['plane0_path'])

for label, sd in loaded_sessions.items():
    print(f"\n{label}: n_cells={sd['n_cells']}, footprints shape={sd['footprints'].shape}, "
          f"centroids shape={sd['centroids'].shape}")


## Function 2.3 — `track_session_group`

The core flexible matching step: register + match ROIs across *any*
caller-chosen small set of sessions, against a caller-chosen reference
among them — generalizes `TrackROIs`'s pairwise-to-one-reference approach
(currently hardcoded to 2 required sessions) to however many sessions you
put in one group.

Built as three pieces:
1. **`_align_session_group_to_reference`** — FOV registration via
   `phase_cross_correlation` (same as `align_fovs`/`apply_shifts`), applied
   to centroids only (footprints get shifted lazily inside matching, same
   as the original).
2. **`_match_two_sessions`** — candidate matching by centroid distance +
   footprint correlation, then Hungarian assignment — the reference-vs-one-
   other-session logic from `match_rois_pairwise`, as a standalone reusable
   piece.
3. **`track_session_group`** — calls 1 once, then 2 once per non-reference
   session, and assembles everything into one registration matrix.

- **Input:** `sessions` (`{label: session_data}` from `load_session_for_tracking`,
  any size), `reference_label`, `max_distance_um=15.0`, `min_correlation=0.3`,
  `microns_per_pixel`.
- **Output:** dict with `registration_matrix` (`n_ref_cells x n_sessions`,
  columns ordered by `labels_order`; `-1` = not found), `labels_order`,
  `reference_label`, `shifts`, `matches` (`{label: [match dicts with
  distance_um/correlation]}`), `sessions` (passed through, for downstream
  filtering/review/plotting).

In [ ]:
def _align_session_group_to_reference(sessions, reference_label):
    """
    FOV registration (translation only) of every session's mean image
    against the reference's, via phase_cross_correlation.

    Returns
    -------
    shifts : dict
        {label: (dy, dx)} in pixels. Reference itself is (0.0, 0.0).
    """
    if reference_label not in sessions:
        raise ValueError(f"reference_label '{reference_label}' not in sessions "
                          f"(available: {list(sessions.keys())})")

    ref_img = sessions[reference_label]['mean_img']
    shifts = {}

    for label, sd in sessions.items():
        if label == reference_label:
            shifts[label] = (0.0, 0.0)
            print(f"  {label}: reference (0, 0)")
            continue

        shift_yx, error, diffphase = phase_cross_correlation(
            ref_img, sd['mean_img'], upsample_factor=10
        )
        shifts[label] = (shift_yx[0], shift_yx[1])
        print(f"  {label}: shift = ({shift_yx[0]:.2f}, {shift_yx[1]:.2f}) pixels")

    return shifts


In [ ]:
def _match_two_sessions(session_a, session_b, shift_a, shift_b,
                         max_distance_um=15.0, min_correlation=0.3,
                         microns_per_pixel=MICRONS_PER_PIXEL):
    """
    Match ROIs from session_a (typically the reference) to session_b:
    candidate pairs by shifted-centroid distance, footprint correlation on
    those candidates, then Hungarian assignment on the surviving pairs.
    Same logic as TrackROIs_SalineDCZ.py's match_rois_pairwise, generalized
    to take pre-shifted centroids via arbitrary shift_a/shift_b rather than
    assuming session_a is always the unshifted reference.

    Parameters
    ----------
    session_a, session_b : dict
        From load_session_for_tracking.
    shift_a, shift_b : tuple of float
        (dy, dx) for each session, from _align_session_group_to_reference.
    max_distance_um, min_correlation : float
    microns_per_pixel : float

    Returns
    -------
    matches : list of dict
        Each: {'idx_a', 'idx_b', 'distance_um', 'correlation'}.
    """
    centroids_a = session_a['centroids'] + np.array(shift_a)
    centroids_b = session_b['centroids'] + np.array(shift_b)

    dist_matrix = cdist(centroids_a, centroids_b, metric='euclidean') * microns_per_pixel

    candidate_pairs = [(i, j) for i in range(dist_matrix.shape[0])
                       for j in range(dist_matrix.shape[1])
                       if dist_matrix[i, j] <= max_distance_um]

    if len(candidate_pairs) == 0:
        return []

    correlations = {}
    for idx_a, idx_b in candidate_pairs:
        fp_a = ndi_shift(session_a['footprints'][idx_a], shift=shift_a, mode='constant', cval=0)
        fp_b = ndi_shift(session_b['footprints'][idx_b], shift=shift_b, mode='constant', cval=0)

        mask = (fp_a > 0) | (fp_b > 0)
        if mask.sum() < 5:
            correlations[(idx_a, idx_b)] = 0.0
            continue

        vals_a, vals_b = fp_a[mask], fp_b[mask]
        if vals_a.std() < 1e-10 or vals_b.std() < 1e-10:
            correlations[(idx_a, idx_b)] = 0.0
            continue

        correlations[(idx_a, idx_b)] = np.corrcoef(vals_a, vals_b)[0, 1]

    valid_pairs = [(i, j) for (i, j) in candidate_pairs
                   if correlations.get((i, j), 0) >= min_correlation]
    if len(valid_pairs) == 0:
        return []

    unique_a = sorted(set(i for i, j in valid_pairs))
    unique_b = sorted(set(j for i, j in valid_pairs))
    map_a = {idx: k for k, idx in enumerate(unique_a)}
    map_b = {idx: k for k, idx in enumerate(unique_b)}

    cost_matrix = np.full((len(unique_a), len(unique_b)), 1e6)
    for i, j in valid_pairs:
        cost_matrix[map_a[i], map_b[j]] = dist_matrix[i, j]

    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    matches = []
    for r, c in zip(row_ind, col_ind):
        if cost_matrix[r, c] < 1e6:
            idx_a, idx_b = unique_a[r], unique_b[c]
            matches.append({
                'idx_a': idx_a,
                'idx_b': idx_b,
                'distance_um': dist_matrix[idx_a, idx_b],
                'correlation': correlations[(idx_a, idx_b)],
            })

    return matches


In [ ]:
def track_session_group(sessions, reference_label, max_distance_um=15.0,
                         min_correlation=0.3, microns_per_pixel=MICRONS_PER_PIXEL):
    """
    Register + match ROIs across an arbitrary caller-chosen group of
    sessions, against a caller-chosen reference session in that group.

    Parameters
    ----------
    sessions : dict
        {label: session_data}, from load_session_for_tracking. Any size --
        typically 2-3 for one scientific comparison.
    reference_label : str
        Must be a key in sessions.
    max_distance_um, min_correlation : float
    microns_per_pixel : float

    Returns
    -------
    tracking_result : dict
        'registration_matrix': (n_ref_cells, n_sessions) int array,
            columns ordered by labels_order, -1 = not found in that session.
        'labels_order': list of str
        'reference_label': str
        'shifts': {label: (dy, dx)}
        'matches': {label: [match dicts]} (non-reference labels only)
        'sessions': the input sessions dict, passed through
    """
    print(f"Aligning {len(sessions)} sessions to reference '{reference_label}'...")
    shifts = _align_session_group_to_reference(sessions, reference_label)

    labels_order = list(sessions.keys())
    ref_col = labels_order.index(reference_label)
    n_ref_cells = sessions[reference_label]['n_cells']

    registration_matrix = np.full((n_ref_cells, len(labels_order)), -1, dtype=int)
    registration_matrix[:, ref_col] = np.arange(n_ref_cells)

    matches = {}
    print(f"\nMatching non-reference sessions...")
    for label in labels_order:
        if label == reference_label:
            continue

        match_list = _match_two_sessions(
            sessions[reference_label], sessions[label],
            shifts[reference_label], shifts[label],
            max_distance_um=max_distance_um, min_correlation=min_correlation,
            microns_per_pixel=microns_per_pixel
        )
        matches[label] = match_list

        col = labels_order.index(label)
        for m in match_list:
            registration_matrix[m['idx_a'], col] = m['idx_b']

        print(f"  {reference_label} vs {label}: {len(match_list)} matches "
              f"(out of {n_ref_cells} reference cells)")

    sessions_per_cell = np.sum(registration_matrix >= 0, axis=1)
    print(f"\nTracked in all {len(labels_order)} sessions: "
          f"{np.sum(sessions_per_cell == len(labels_order))} / {n_ref_cells} reference cells")

    return {
        'registration_matrix': registration_matrix,
        'labels_order': labels_order,
        'reference_label': reference_label,
        'shifts': shifts,
        'matches': matches,
        'sessions': sessions,
    }


In [ ]:
# --- Try it on the two real sessions already loaded (Day1 as reference) ---
tracking_result = track_session_group(loaded_sessions, reference_label='Day1')

print("\nregistration_matrix shape:", tracking_result['registration_matrix'].shape)
print("labels_order:", tracking_result['labels_order'])


## Function 2.4 — `filter_and_flag_matches`

Of the cells tracked across *every* session in the group, sorts each into:
- **`auto_reject`** — footprint size in any session exceeds
  `max_footprint_pixels` (a segmentation artifact/merged-ROI red flag)
- **`auto_accept`** — footprint correlation and centroid distance vs. the
  reference are within threshold (`auto_accept_corr`/`auto_accept_dist`)
  in *every* non-reference session
- **`manual_review`** — everything else, i.e. plausible but not confidently
  automatic

Same logic as `TrackROIs_SalineDCZ.py`'s `filter_registration_matrix`, but
generalized from a hardcoded 2-session pair to "worst across however many
non-reference sessions are in this group."

- **Input:** `tracking_result` (from `track_session_group`),
  `max_footprint_pixels=500`, `auto_accept_corr=0.7`, `auto_accept_dist=5.0`,
  `microns_per_pixel`.
- **Output:** `filtered_matrix` (registration matrix with auto-rejected rows
  zeroed to `-1`), `cell_categories` (`{row: category}`), `filter_report`
  (summary counts + reject/accept details).

In [ ]:
def filter_and_flag_matches(tracking_result, max_footprint_pixels=500,
                             auto_accept_corr=0.7, auto_accept_dist=5.0,
                             microns_per_pixel=MICRONS_PER_PIXEL):
    """
    Categorize each cell tracked across every session in the group into
    auto_accept / auto_reject / manual_review. See markdown above.

    Parameters
    ----------
    tracking_result : dict
        From track_session_group.
    max_footprint_pixels : int
    auto_accept_corr : float
    auto_accept_dist : float
    microns_per_pixel : float

    Returns
    -------
    filtered_matrix : numpy.ndarray
    cell_categories : dict
        {row: 'auto_accept' | 'auto_reject' | 'manual_review'}
    filter_report : dict
    """
    registration_matrix = tracking_result['registration_matrix']
    labels_order = tracking_result['labels_order']
    reference_label = tracking_result['reference_label']
    shifts = tracking_result['shifts']
    sessions = tracking_result['sessions']

    ref_col = labels_order.index(reference_label)
    non_ref_labels = [l for l in labels_order if l != reference_label]

    mask = np.all(registration_matrix >= 0, axis=1)
    candidate_rows = np.where(mask)[0]

    filtered_matrix = registration_matrix.copy()
    cell_categories = {}
    reject_details = []
    accept_details = []

    print(f"Evaluating {len(candidate_rows)} cells tracked across all "
          f"{len(labels_order)} sessions...")

    for row in candidate_rows:
        # --- Footprint-size artifact check, every session ---
        is_artifact = False
        for col, label in enumerate(labels_order):
            roi_idx = registration_matrix[row, col]
            fp_size = np.sum(sessions[label]['footprints'][roi_idx] > 0)
            if fp_size > max_footprint_pixels:
                is_artifact = True
                reject_details.append({'row': int(row), 'label': label,
                                        'roi_idx': int(roi_idx), 'fp_size': int(fp_size)})
                break

        if is_artifact:
            filtered_matrix[row, :] = -1
            cell_categories[row] = 'auto_reject'
            continue

        # --- Worst correlation/distance vs reference, every non-reference session ---
        ref_roi = registration_matrix[row, ref_col]
        fp_ref = ndi_shift(sessions[reference_label]['footprints'][ref_roi],
                            shift=shifts[reference_label], mode='constant', cval=0)
        cent_ref = sessions[reference_label]['centroids'][ref_roi] + np.array(shifts[reference_label])

        all_high_quality = True
        worst_corr, worst_dist = 1.0, 0.0

        for label in non_ref_labels:
            col = labels_order.index(label)
            roi_idx = registration_matrix[row, col]

            cent_other = sessions[label]['centroids'][roi_idx] + np.array(shifts[label])
            dist_um = np.linalg.norm(cent_ref - cent_other) * microns_per_pixel

            fp_other = ndi_shift(sessions[label]['footprints'][roi_idx],
                                  shift=shifts[label], mode='constant', cval=0)
            union_mask = (fp_ref > 0) | (fp_other > 0)
            if union_mask.sum() < 5:
                all_high_quality = False
                break

            vals_a, vals_b = fp_ref[union_mask], fp_other[union_mask]
            if vals_a.std() < 1e-10 or vals_b.std() < 1e-10:
                all_high_quality = False
                break

            corr = np.corrcoef(vals_a, vals_b)[0, 1]
            worst_corr = min(worst_corr, corr)
            worst_dist = max(worst_dist, dist_um)

            if corr < auto_accept_corr or dist_um > auto_accept_dist:
                all_high_quality = False
                break

        if all_high_quality:
            cell_categories[row] = 'auto_accept'
            accept_details.append({'row': int(row), 'worst_corr': worst_corr, 'worst_dist': worst_dist})
        else:
            cell_categories[row] = 'manual_review'

    n_accept = sum(1 for v in cell_categories.values() if v == 'auto_accept')
    n_reject = sum(1 for v in cell_categories.values() if v == 'auto_reject')
    n_manual = sum(1 for v in cell_categories.values() if v == 'manual_review')

    print(f"\n  Auto-accept:   {n_accept}")
    print(f"  Auto-reject:   {n_reject}  (footprint > {max_footprint_pixels}px in some session)")
    print(f"  Manual review: {n_manual}")

    filter_report = {
        'n_candidates': len(candidate_rows),
        'n_auto_accept': n_accept,
        'n_auto_reject': n_reject,
        'n_manual_review': n_manual,
        'reject_details': reject_details,
        'accept_details': accept_details,
    }

    return filtered_matrix, cell_categories, filter_report


In [ ]:
# --- Try it on the real 3-session tracking result ---
filtered_matrix, cell_categories, filter_report = filter_and_flag_matches(tracking_result)


## Dev checkpoint utility (not part of the official Phase 2 outline)

Testing convenience only: save/reload the tracking+filter results for one
session group so you don't have to re-run alignment/matching/filtering
every time you reopen the notebook. Skips pickling
`tracking_result['sessions']` (the large dense footprint arrays) --
instead saves each session's `plane0_path` and reconstructs `sessions`
fresh via `load_session_for_tracking` on load, since that's fast and keeps
the checkpoint file small.

- **Save input:** `tracking_result`, `cell_categories`, `filter_report`,
  `plane0_paths` (`{label: plane0_path}`, e.g. built from `session_catalog`),
  `checkpoint_path`.
- **Load input:** `checkpoint_path`, `reload_sessions=True`.
- **Load output:** `tracking_result`, `cell_categories`, `filter_report` --
  same shapes as what produced the checkpoint.

In [ ]:
import pickle


def save_tracking_checkpoint(tracking_result, cell_categories, filter_report,
                              plane0_paths, checkpoint_path):
    """
    Save the lightweight parts of a tracking run to disk, skipping the
    large footprint arrays (cheap to rebuild from suite2p output instead).
    """
    checkpoint = {
        'registration_matrix': tracking_result['registration_matrix'],
        'labels_order': tracking_result['labels_order'],
        'reference_label': tracking_result['reference_label'],
        'shifts': tracking_result['shifts'],
        'matches': tracking_result['matches'],
        'cell_categories': cell_categories,
        'filter_report': filter_report,
        'plane0_paths': {label: str(plane0_paths[label]) for label in tracking_result['labels_order']},
    }
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint, f)
    print(f"Saved checkpoint -> {checkpoint_path}")


def load_tracking_checkpoint(checkpoint_path, reload_sessions=True):
    """
    Reload a checkpoint saved by save_tracking_checkpoint. Rebuilds
    tracking_result['sessions'] fresh from suite2p output if reload_sessions
    (fast -- just re-reads stat/iscell/ops.npy), rather than having pickled
    the large footprint arrays.
    """
    with open(checkpoint_path, 'rb') as f:
        checkpoint = pickle.load(f)

    sessions = {}
    if reload_sessions:
        print("Reloading sessions from suite2p output...")
        for label, plane0_path in checkpoint['plane0_paths'].items():
            sessions[label] = load_session_for_tracking(plane0_path)

    tracking_result = {
        'registration_matrix': checkpoint['registration_matrix'],
        'labels_order': checkpoint['labels_order'],
        'reference_label': checkpoint['reference_label'],
        'shifts': checkpoint['shifts'],
        'matches': checkpoint['matches'],
        'sessions': sessions,
    }

    print(f"Loaded checkpoint from {checkpoint_path}")
    return tracking_result, checkpoint['cell_categories'], checkpoint['filter_report']


In [ ]:
# --- Save your current real result so you don't have to redo it after a restart ---
plane0_paths = {label: session_catalog[label]['plane0_path'] for label in tracking_result['labels_order']}
save_tracking_checkpoint(tracking_result, cell_categories, filter_report, plane0_paths, checkpoint_path)


In [ ]:
# --- After a kernel restart, skip straight to here instead of re-running everything above ---
tracking_result, cell_categories, filter_report = load_tracking_checkpoint(checkpoint_path)

print("\nlabels_order:", tracking_result['labels_order'])
print("registration_matrix shape:", tracking_result['registration_matrix'].shape)
print("filter_report:", {k: v for k, v in filter_report.items() if k in
                          ('n_candidates', 'n_auto_accept', 'n_auto_reject', 'n_manual_review')})


## Function 2.5 — `manual_review_matches`

Interactive keyboard review for every cell `filter_and_flag_matches` put in
the `manual_review` bucket. Built as two pieces:

1. **`plot_cell_across_group`** — one figure per cell: a footprint-crop
   panel per session (green overlay on that session's aligned mean image,
   titled with correlation/distance vs. the reference) plus a final
   "all overlaid" panel colored by session. Generalizes
   `TrackROIs_SalineDCZ.py`'s `plot_cell_across_sessions` from a fixed
   `required_days` pair to however many sessions are in the group.
2. **`manual_review_matches`** — the review loop: `'a'`/Right = accept,
   `'r'`/`'x'` = reject, `'b'`/Left = back, `'q'` = quit early. Waits for
   each keypress via `fig.canvas.start_event_loop()` polling (the pattern
   fixed in Phase 1's review popup) rather than `TrackROIs`'s original
   `plt.show(block=False)`, which hit the same Jupyter/Qt rendering race we
   diagnosed there. Opens/closes a fresh figure per cell rather than
   redrawing into one persistent window — a little more flicker between
   cells, but avoids re-introducing that same race for the sake of a
   marginal UI nicety.

- **Input:** `tracking_result`, `cell_categories` (from
  `filter_and_flag_matches`), `crop_size=40`, `microns_per_pixel`.
- **Output:** `verified_matrix` (registration matrix with rejected rows set
  to `-1`), `decisions` (`{row: 'accept' | 'reject'}` for every row you
  actually reviewed before quitting/finishing).

In [ ]:
def plot_cell_across_group(tracking_result, cell_row, crop_size=40,
                            microns_per_pixel=MICRONS_PER_PIXEL):
    """
    One figure for a single tracked cell: a footprint-crop panel per
    session in the group, plus an "all overlaid" panel colored by session.

    Parameters
    ----------
    tracking_result : dict
        From track_session_group.
    cell_row : int
        Row index into registration_matrix (a reference-session ROI index).
    crop_size : int
        Half-width of the crop window, in pixels.
    microns_per_pixel : float

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    registration_matrix = tracking_result['registration_matrix']
    labels_order = tracking_result['labels_order']
    reference_label = tracking_result['reference_label']
    shifts = tracking_result['shifts']
    sessions = tracking_result['sessions']

    ref_col = labels_order.index(reference_label)
    ref_roi = registration_matrix[cell_row, ref_col]
    fp_ref = ndi_shift(sessions[reference_label]['footprints'][ref_roi],
                        shift=shifts[reference_label], mode='constant', cval=0)
    cent_ref = sessions[reference_label]['centroids'][ref_roi] + np.array(shifts[reference_label])

    corrs, dists, all_cents = {}, {}, []
    for label in labels_order:
        col = labels_order.index(label)
        roi_idx = registration_matrix[cell_row, col]
        cent = sessions[label]['centroids'][roi_idx] + np.array(shifts[label])
        all_cents.append(cent)
        dists[label] = np.linalg.norm(cent_ref - cent) * microns_per_pixel

        if label == reference_label:
            corrs[label] = 1.0
            continue

        fp_other = ndi_shift(sessions[label]['footprints'][roi_idx],
                              shift=shifts[label], mode='constant', cval=0)
        union_mask = (fp_ref > 0) | (fp_other > 0)
        if union_mask.sum() < 5:
            corrs[label] = 0.0
            continue
        vals_a, vals_b = fp_ref[union_mask], fp_other[union_mask]
        if vals_a.std() < 1e-10 or vals_b.std() < 1e-10:
            corrs[label] = 0.0
            continue
        corrs[label] = np.corrcoef(vals_a, vals_b)[0, 1]

    all_cents = np.array(all_cents)
    mid_y, mid_x = int(np.mean(all_cents[:, 0])), int(np.mean(all_cents[:, 1]))

    Ly, Lx = sessions[reference_label]['Ly'], sessions[reference_label]['Lx']
    y_min, y_max = max(0, mid_y - crop_size), min(Ly, mid_y + crop_size)
    x_min, x_max = max(0, mid_x - crop_size), min(Lx, mid_x + crop_size)

    n_sessions = len(labels_order)
    fig, axes = plt.subplots(1, n_sessions + 1, figsize=(4 * (n_sessions + 1), 4))

    footprints_cropped = []
    for i, label in enumerate(labels_order):
        col = labels_order.index(label)
        roi_idx = registration_matrix[cell_row, col]

        fp = ndi_shift(sessions[label]['footprints'][roi_idx],
                        shift=shifts[label], mode='constant', cval=0)
        mean_shifted = ndi_shift(sessions[label]['mean_img'],
                                  shift=shifts[label], mode='constant', cval=0)

        # ndi_shift's default cubic-spline interpolation can introduce tiny
        # negative overshoot near sharp edges even for a non-negative input
        # (footprint values are all >= 0) -- clip it out before normalizing,
        # so imshow doesn't warn about out-of-range RGBA values below.
        fp_crop = np.clip(fp[y_min:y_max, x_min:x_max], 0, None)
        mean_crop = mean_shifted[y_min:y_max, x_min:x_max]
        footprints_cropped.append(fp_crop)

        fp_norm = fp_crop / fp_crop.max() if fp_crop.max() > 0 else fp_crop
        axes[i].imshow(mean_crop, cmap='gray')
        overlay = np.zeros((*fp_crop.shape, 4))
        overlay[:, :, 1] = fp_norm
        overlay[:, :, 3] = fp_norm * 0.6
        axes[i].imshow(overlay)

        if label == reference_label:
            axes[i].set_title(f"{label} (reference)\nROI {roi_idx}", fontsize=9)
        else:
            axes[i].set_title(f"{label}\nROI {roi_idx}\n"
                               f"corr={corrs[label]:.2f}, dist={dists[label]:.1f}um", fontsize=9)
        axes[i].axis('off')

    colors = plt.cm.hsv(np.linspace(0, 0.85, n_sessions))
    ref_mean_shifted = ndi_shift(sessions[reference_label]['mean_img'],
                                  shift=shifts[reference_label], mode='constant', cval=0)
    mean_crop_ref = ref_mean_shifted[y_min:y_max, x_min:x_max]

    axes[n_sessions].imshow(mean_crop_ref, cmap='gray')
    overlay_all = np.zeros((*footprints_cropped[0].shape, 4))
    for j, fp_crop in enumerate(footprints_cropped):
        fp_norm = fp_crop / fp_crop.max() if fp_crop.max() > 0 else fp_crop
        mask = fp_norm > 0.1
        overlay_all[mask, 0] += colors[j, 0] * fp_norm[mask]
        overlay_all[mask, 1] += colors[j, 1] * fp_norm[mask]
        overlay_all[mask, 2] += colors[j, 2] * fp_norm[mask]
        overlay_all[mask, 3] = np.maximum(overlay_all[mask, 3], fp_norm[mask] * 0.6)
    overlay_all[:, :, :3] = np.clip(overlay_all[:, :, :3], 0, 1)
    axes[n_sessions].imshow(overlay_all)

    non_ref = [l for l in labels_order if l != reference_label]
    if non_ref:
        min_corr = min(corrs[l] for l in non_ref)
        max_dist = max(dists[l] for l in non_ref)
        axes[n_sessions].set_title(f"All overlaid\nmin corr={min_corr:.2f}\n"
                                    f"max dist={max_dist:.1f}um", fontsize=9)
    else:
        axes[n_sessions].set_title("All overlaid", fontsize=9)
    axes[n_sessions].axis('off')

    fig.suptitle(f"Cell row {cell_row} (reference: {reference_label} ROI {ref_roi})",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    return fig


In [ ]:
def manual_review_matches(tracking_result, cell_categories, crop_size=40,
                           microns_per_pixel=MICRONS_PER_PIXEL):
    """
    Interactive keyboard review for every row categorized 'manual_review'.
    'a'/Right = accept, 'r'/'x' = reject, 'b'/Left = back, 'q' = quit early.

    Parameters
    ----------
    tracking_result : dict
        From track_session_group.
    cell_categories : dict
        From filter_and_flag_matches.
    crop_size : int
    microns_per_pixel : float

    Returns
    -------
    verified_matrix : numpy.ndarray
        registration_matrix restricted to auto_accept rows plus
        manually-accepted rows -- auto_reject, manually-rejected, AND any
        manual_review row you didn't get to before quitting are all
        excluded (not "verified" just because you ran out of time).
    decisions : dict
        {row: 'accept' | 'reject'} for every row actually reviewed.
    """
    manual_rows = sorted([row for row, cat in cell_categories.items() if cat == 'manual_review'])
    total = len(manual_rows)

    print(f"Manual review: {total} cells")
    print("Controls: 'a'/Right = accept, 'r'/'x' = reject, 'b'/Left = back, 'q' = quit\n")

    decisions = {}
    idx = 0

    while 0 <= idx < total:
        row = manual_rows[idx]
        fig = plot_cell_across_group(tracking_result, row, crop_size=crop_size,
                                      microns_per_pixel=microns_per_pixel)

        status = decisions.get(row, None)
        status_str = "NOT REVIEWED" if status is None else status.upper()
        fig.suptitle(f"Manual Review {idx+1}/{total} (row {row}) | Status: {status_str}\n"
                     "Click figure, then 'a'=accept  'r'=reject  'b'=back  'q'=quit",
                     fontsize=11, fontweight='bold')

        action = {'value': None}

        def on_key(event):
            if event.key in ('a', 'right'):
                action['value'] = 'accept'
                fig.canvas.stop_event_loop()
            elif event.key in ('r', 'x'):
                action['value'] = 'reject'
                fig.canvas.stop_event_loop()
            elif event.key in ('b', 'left'):
                action['value'] = 'back'
                fig.canvas.stop_event_loop()
            elif event.key == 'q':
                action['value'] = 'quit'
                fig.canvas.stop_event_loop()

        fig.canvas.mpl_connect('key_press_event', on_key)
        plt.show(block=False)

        while action['value'] is None and plt.fignum_exists(fig.number):
            fig.canvas.start_event_loop(0.1)

        if plt.fignum_exists(fig.number):
            plt.close(fig)

        if action['value'] in ('accept', 'reject'):
            decisions[row] = action['value']
            idx += 1
        elif action['value'] == 'back':
            idx = max(0, idx - 1)
        else:  # 'quit' or window closed without a keypress
            print("\nQuitting review.")
            break

    # "Verified" = auto_accept rows, plus manually-accepted rows. Everything
    # else (auto_reject, manually-rejected, and any manual_review row not
    # reached before quitting) is excluded -- being unreviewed is not the
    # same as being accepted.
    verified_matrix = tracking_result['registration_matrix'].copy()
    for row, category in cell_categories.items():
        if category == 'auto_reject':
            verified_matrix[row, :] = -1
        elif category == 'manual_review':
            if decisions.get(row) != 'accept':
                verified_matrix[row, :] = -1
        # 'auto_accept' rows: left as-is (already valid in the matrix)

    n_accept = sum(1 for d in decisions.values() if d == 'accept')
    n_reject = sum(1 for d in decisions.values() if d == 'reject')
    n_unreviewed = total - len(decisions)

    print(f"\n  Manually accepted: {n_accept}")
    print(f"  Manually rejected: {n_reject}")
    print(f"  Unreviewed:        {n_unreviewed}  (excluded from verified_matrix, not accepted)")

    return verified_matrix, decisions


In [ ]:
# --- Try it on the real manual-review bucket (184 cells) ---
# You don't need to review all 184 to test this -- review a handful, then
# press 'q' to quit early and check the partial decisions/verified_matrix.
verified_matrix, decisions = manual_review_matches(tracking_result, cell_categories)


## Function 2.6 — `save_tracking_group`

The real Phase 2 pipeline output (distinct from the dev checkpoint above,
which is a testing convenience) — saved as `{group_name}_tracking_results.h5`
in a `TrackedGroups/` subfolder of the reference session's TSeries
directory, mirroring where `TrackROIs_SalineDCZ.py` saves
`roi_tracking_results.h5` (a `TrackedROIs/` subfolder next to the sessions).

Saves the **verified** matrix (post auto-filter + manual review), the
original unfiltered matrix (for audit), `labels_order`/`reference_label`,
per-session `plane0_path` (so downstream steps can reload `sessions` without
re-discovering them), `shifts`, and the `decisions` dict from manual review.

- **Input:** `tracking_result`, `verified_matrix`, `decisions`,
  `filter_report`, `plane0_paths`, `group_name` (e.g.
  `"BaselineDay1_vs_SalineDCZ1"`), `save_dir` (defaults to a `TrackedGroups`
  folder next to the reference session).
- **Output:** `save_path` (the file written).

In [ ]:
def save_tracking_group(tracking_result, verified_matrix, decisions, filter_report,
                         plane0_paths, group_name, save_dir=None):
    """
    Save one session-group's verified tracking result to HDF5. See markdown
    above for what's saved and where.

    Parameters
    ----------
    tracking_result : dict
        From track_session_group.
    verified_matrix : numpy.ndarray
        Post auto-filter + manual review (from manual_review_matches, or
        just filtered_matrix if you skipped manual review).
    decisions : dict
        From manual_review_matches (may be empty).
    filter_report : dict
        From filter_and_flag_matches.
    plane0_paths : dict
        {label: plane0_path}.
    group_name : str
        e.g. "BaselineDay1_vs_SalineDCZ1".
    save_dir : str, optional
        Defaults to a 'TrackedGroups' folder next to the reference session's
        TSeries directory.

    Returns
    -------
    save_path : str
    """
    labels_order = tracking_result['labels_order']
    reference_label = tracking_result['reference_label']
    shifts = tracking_result['shifts']
    registration_matrix = tracking_result['registration_matrix']

    if save_dir is None:
        ref_tseries_dir = os.path.dirname(os.path.dirname(str(plane0_paths[reference_label])))
        save_dir = os.path.join(ref_tseries_dir, 'TrackedGroups')
    os.makedirs(save_dir, exist_ok=True)

    save_path = os.path.join(save_dir, f'{group_name}_tracking_results.h5')

    with h5py.File(save_path, 'w') as f:
        f.create_dataset('verified_matrix', data=verified_matrix)
        f.create_dataset('original_matrix', data=registration_matrix)

        f.attrs['group_name'] = group_name
        f.attrs['labels_order'] = labels_order
        f.attrs['reference_label'] = reference_label

        shift_grp = f.create_group('shifts')
        for label, (dy, dx) in shifts.items():
            shift_grp.attrs[label] = [dy, dx]

        plane0_grp = f.create_group('plane0_paths')
        for label in labels_order:
            plane0_grp.attrs[label] = str(plane0_paths[label])

        if decisions:
            dec_grp = f.create_group('decisions')
            rows = list(decisions.keys())
            vals = [decisions[r] for r in rows]
            dec_grp.create_dataset('rows', data=np.array(rows, dtype=int))
            dec_grp.create_dataset('decisions', data=np.array(vals, dtype='S10'))

        report_grp = f.create_group('filter_report')
        for k in ('n_candidates', 'n_auto_accept', 'n_auto_reject', 'n_manual_review'):
            if k in filter_report:
                report_grp.attrs[k] = filter_report[k]

    n_tracked = np.sum(np.all(verified_matrix >= 0, axis=1))
    print(f"Saved tracking group '{group_name}' -> {save_path}")
    print(f"  Sessions: {labels_order}  (reference: {reference_label})")
    print(f"  Final tracked cells: {n_tracked}")

    return save_path


def _decode(x):
    """h5py attrs may come back as bytes depending on version -- normalize to str."""
    return x.decode() if isinstance(x, bytes) else x


def load_tracking_group(save_path):
    """
    Reload a tracking group saved by save_tracking_group.

    Returns
    -------
    result : dict with keys:
        verified_matrix, original_matrix, labels_order, reference_label,
        shifts, plane0_paths, decisions, filter_report.
    """
    with h5py.File(save_path, 'r') as f:
        verified_matrix = f['verified_matrix'][:]
        original_matrix = f['original_matrix'][:]
        labels_order = [_decode(l) for l in f.attrs['labels_order']]
        reference_label = _decode(f.attrs['reference_label'])

        shifts = {label: tuple(f['shifts'].attrs[label]) for label in labels_order}
        plane0_paths = {label: _decode(f['plane0_paths'].attrs[label]) for label in labels_order}

        decisions = {}
        if 'decisions' in f:
            rows = f['decisions']['rows'][:]
            vals = [_decode(v) for v in f['decisions']['decisions'][:]]
            decisions = {int(r): v for r, v in zip(rows, vals)}

        filter_report = {k: v for k, v in f['filter_report'].attrs.items()}

    return {
        'verified_matrix': verified_matrix,
        'original_matrix': original_matrix,
        'labels_order': labels_order,
        'reference_label': reference_label,
        'shifts': shifts,
        'plane0_paths': plane0_paths,
        'decisions': decisions,
        'filter_report': filter_report,
    }


In [ ]:
# --- Save your real tracking group and verify the round-trip ---
plane0_paths = {label: session_catalog[label]['plane0_path'] for label in tracking_result['labels_order']}
group_name = 'Day1_vs_SalineDCZ1'

group_save_path = save_tracking_group(
    tracking_result, verified_matrix, decisions, filter_report,
    plane0_paths, group_name
)

reloaded_group = load_tracking_group(group_save_path)
print("\nRound-trip check:")
print("  labels_order match:", reloaded_group['labels_order'] == tracking_result['labels_order'])
print("  reference_label match:", reloaded_group['reference_label'] == tracking_result['reference_label'])
print("  verified_matrix match:", np.array_equal(reloaded_group['verified_matrix'], verified_matrix))
print("  decisions match:", reloaded_group['decisions'] == decisions)


## Function 2.7 — `build_master_cell_table`

Joins a verified tracking-group matrix with each session's Phase 1 layer
labels into the actual master table. Built as three pieces:

1. **`find_layer_curve_path`** — locate a session's
   `*_layer_curve_results.h5` from its `plane0_path`, same glob-based
   discovery convention as `MergeTrackedLayerSMI.py`'s
   `find_smi_results_path` — so you don't have to hand-type these paths.
2. **`_load_layer_of_cell`** — load just the `{roi_idx: layer_name}` mapping
   out of one of those files. Reimplemented rather than imported from
   `1.LayerAssignment_Curve.py` — that module's filename starts with a
   digit, which Python can't `import` via a normal statement anyway.
3. **`_worst_corr_dist_for_row`** — the same worst-correlation/largest-
   distance-vs-reference computation `filter_and_flag_matches` uses,
   factored out so the master table can report match confidence for every
   row (auto-accepted or manually-accepted alike), not just the rows that
   happened to go through the auto-accept path.
4. **`build_master_cell_table`** — assembles all of the above into one
   `pandas.DataFrame`.

- **Input:** `tracking_result` (from `track_session_group`),
  `verified_matrix` (from `manual_review_matches`, or `filtered_matrix` if
  you skipped manual review), `layer_curve_paths` (`{label: path}`, e.g.
  built with `find_layer_curve_path`), `microns_per_pixel`.
- **Output:** `df` — one row per verified tracked cell:
  `global_cell_id`, `roi_idx_<label>` / `layer_<label>` per session,
  `worst_corr`, `worst_dist_um`.

In [ ]:
def find_layer_curve_path(plane0_path, prefer_averaged=True):
    """
    Locate a session's Phase 1 layer-curve-results file, given its
    suite2p/plane0 path -- same glob-based discovery convention as
    MergeTrackedLayerSMI.py's find_smi_results_path.

    Two Phase 1 outputs can coexist per session: the retired
    independent-click version's output (*_layer_curve_results.h5) and the
    current 1.LayerAssignment_Curve.py's chained-registration/averaged
    output (*_layer_curve_results_averaged.h5, kept under that suffix even
    now that it's the only Phase 1 script, specifically so it never
    overwrites the independent-click results already computed for real) --
    the two glob patterns below are naturally disjoint (the averaged
    filename doesn't end in '_layer_curve_results.h5'), so this never
    confuses the two. Prefers the averaged variant by default since that's
    the current script's output; pass prefer_averaged=False to force the
    retired version's if you need to compare against it.
    """
    tseries_dir = os.path.dirname(os.path.dirname(str(plane0_path)))
    averaged_matches = glob.glob(os.path.join(tseries_dir, '*_layer_curve_results_averaged.h5'))
    independent_matches = glob.glob(os.path.join(tseries_dir, '*_layer_curve_results.h5'))

    if prefer_averaged and averaged_matches:
        matches = averaged_matches
    elif independent_matches:
        matches = independent_matches
    elif averaged_matches:
        matches = averaged_matches
    else:
        raise FileNotFoundError(f"No *_layer_curve_results(_averaged).h5 found in {tseries_dir} "
                                 "-- has Phase 1 been run for this session?")

    if len(matches) > 1:
        print(f"WARNING: multiple matches in {tseries_dir}, using {matches[0]}")
    return matches[0]


def _load_layer_of_cell(layer_curve_path):
    """
    Load the {roi_idx: layer_name} mapping from a Phase 1
    *_layer_curve_results.h5 file, keyed by POSITION within the
    iscell-filtered cell list (0-based) -- matching Phase 2's own roi_idx
    convention in registration_matrix/verified_matrix (built from
    load_session_for_tracking's footprints[i]/centroids[i], which are
    indexed by enumerate() position, not by raw suite2p ROI index).

    BUG FIX: this used to key the dict by the file's own 'cell_idx' dataset
    (the RAW suite2p stat.npy index, e.g. 3, 17, 108... after most raw
    detections get filtered out by iscell) instead of by position. Since
    Phase 2's roi_idx is a position, not a raw index, that lookup was
    wrong for nearly every cell -- coincidental collisions gave silently
    WRONG layer labels, and non-collisions gave spurious None values. Both
    Phase 1 and Phase 2 build their iscell-filtered lists the same way
    (np.where(iscell[:,0]==1)[0]) from the same iscell.npy, so position i
    refers to the same physical cell in both -- that's the correct join key.
    """
    with h5py.File(layer_curve_path, 'r') as f:
        layer_codes = f['layer_codes'][:]
        layer_names = tuple(n.decode() if isinstance(n, bytes) else n
                             for n in f['layer_names'][:])
    return {
        position: layer_names[code]
        for position, code in enumerate(layer_codes) if code >= 0
    }


def _worst_corr_dist_for_row(tracking_result, row, microns_per_pixel=MICRONS_PER_PIXEL):
    """
    Worst footprint correlation / largest centroid distance (vs reference)
    across all non-reference sessions, for one tracked cell -- same
    computation filter_and_flag_matches uses, factored out for reuse.
    """
    registration_matrix = tracking_result['registration_matrix']
    labels_order = tracking_result['labels_order']
    reference_label = tracking_result['reference_label']
    shifts = tracking_result['shifts']
    sessions = tracking_result['sessions']

    ref_col = labels_order.index(reference_label)
    ref_roi = registration_matrix[row, ref_col]
    fp_ref = ndi_shift(sessions[reference_label]['footprints'][ref_roi],
                        shift=shifts[reference_label], mode='constant', cval=0)
    cent_ref = sessions[reference_label]['centroids'][ref_roi] + np.array(shifts[reference_label])

    worst_corr, worst_dist = 1.0, 0.0
    for label in labels_order:
        if label == reference_label:
            continue
        col = labels_order.index(label)
        roi_idx = registration_matrix[row, col]

        cent_other = sessions[label]['centroids'][roi_idx] + np.array(shifts[label])
        dist_um = np.linalg.norm(cent_ref - cent_other) * microns_per_pixel

        fp_other = ndi_shift(sessions[label]['footprints'][roi_idx],
                              shift=shifts[label], mode='constant', cval=0)
        union_mask = (fp_ref > 0) | (fp_other > 0)
        if union_mask.sum() < 5:
            corr = 0.0
        else:
            vals_a, vals_b = fp_ref[union_mask], fp_other[union_mask]
            corr = 0.0 if (vals_a.std() < 1e-10 or vals_b.std() < 1e-10) else np.corrcoef(vals_a, vals_b)[0, 1]

        worst_corr = min(worst_corr, corr)
        worst_dist = max(worst_dist, dist_um)

    return worst_corr, worst_dist


def build_master_cell_table(tracking_result, verified_matrix, layer_curve_paths,
                             microns_per_pixel=MICRONS_PER_PIXEL):
    """
    Join a verified tracking-group registration matrix with each session's
    Phase 1 layer labels into one tidy table.

    Parameters
    ----------
    tracking_result : dict
        From track_session_group.
    verified_matrix : numpy.ndarray
        Post auto-filter + manual review.
    layer_curve_paths : dict
        {label: path to that session's *_layer_curve_results.h5}.
    microns_per_pixel : float

    Returns
    -------
    df : pandas.DataFrame
        One row per verified tracked cell: global_cell_id, then per-session
        roi_idx_<label>/layer_<label>, plus worst_corr, worst_dist_um.
    """
    labels_order = tracking_result['labels_order']

    layer_lookup = {label: _load_layer_of_cell(layer_curve_paths[label]) for label in labels_order}

    mask = np.all(verified_matrix >= 0, axis=1)
    tracked_rows = np.where(mask)[0]
    print(f"Building master table for {len(tracked_rows)} verified tracked cells...")

    records = []
    for row in tracked_rows:
        record = {'global_cell_id': int(row)}
        for col, label in enumerate(labels_order):
            roi_idx = int(verified_matrix[row, col])
            record[f'roi_idx_{label}'] = roi_idx
            record[f'layer_{label}'] = layer_lookup[label].get(roi_idx, None)

        worst_corr, worst_dist = _worst_corr_dist_for_row(tracking_result, row, microns_per_pixel)
        record['worst_corr'] = worst_corr
        record['worst_dist_um'] = worst_dist
        records.append(record)

    df = pd.DataFrame.from_records(records)

    layer_cols = [c for c in df.columns if c.startswith('layer_')]
    n_layer_mismatch = df[layer_cols].nunique(axis=1).gt(1).sum()
    print(f"  {len(df)} cells in master table.")
    print(f"  {n_layer_mismatch} cells have differing layer labels across sessions "
          f"(flagged for Function 2.8, flag_layer_mismatches).")

    return df


In [ ]:
# --- Try it: build the master table for your real Day1/SalineDCZ1 group ---
# Requires tracking_result + verified_matrix to already exist for this group
# on the REAL animal folder (re-run discover -> load -> track -> filter ->
# manual_review_matches above first if you haven't yet on the real data).

layer_curve_paths = {
    label: find_layer_curve_path(session_catalog[label]['plane0_path'])
    for label in tracking_result['labels_order']
}
print("layer_curve_paths:", layer_curve_paths)

master_df = build_master_cell_table(tracking_result, verified_matrix, layer_curve_paths)
master_df.head(10)


### Fix — `save_master_cell_table` / `load_master_cell_table`

`track_and_build_table`'s `save_tracking_group` call only persists the raw
tracking matrix/decisions/filter_report — `master_df` (the joined
tracked-cell + layer table Track A's `join_smi_to_master_table` actually
needs) was only ever returned in-memory, never written to disk. Every
time Track A work resumes, this would force re-running tracking from
scratch just to regenerate a table that was already built and reviewed
once.

Saves `master_df` as `{group_name}_master_table.csv` in the same
`TrackedGroups/` folder `save_tracking_group` already uses (so both files
sit together), plus the mismatch-review `resolutions` dict (if any) as
`{group_name}_mismatch_resolutions.json` — those decisions were being lost
too.

Wired into `track_and_build_table` below (Function 2.10), called
automatically once `master_df`/`resolutions` are both available — no
separate step needed going forward.

- **Input:** `master_df`, `group_name`, `save_dir` (same directory
  `save_tracking_group` used — pass `os.path.dirname(group_save_path)`),
  `resolutions` (optional).
- **Output:** `saved_paths` (`{'table': path, 'resolutions': path or None}`).

In [ ]:
def save_master_cell_table(master_df, group_name, save_dir, resolutions=None):
    """
    Save one tracking group's master_df to '{group_name}_master_table.csv'
    in save_dir, alongside save_tracking_group's
    '{group_name}_tracking_results.h5'. Also saves the mismatch-review
    resolutions (if any) as '{group_name}_mismatch_resolutions.json'. See
    markdown above for why this exists.

    Parameters
    ----------
    master_df : pandas.DataFrame
        From build_master_cell_table (optionally through flag_layer_mismatches).
    group_name : str
    save_dir : str
        Same directory save_tracking_group used (its return value's dirname).
    resolutions : dict, optional
        From review_layer_mismatches_popup.

    Returns
    -------
    saved_paths : dict
        {'table': path, 'resolutions': path or None}.
    """
    os.makedirs(save_dir, exist_ok=True)

    table_path = os.path.join(save_dir, f'{group_name}_master_table.csv')
    master_df.to_csv(table_path, index=False)
    print(f"Saved master table -> {table_path}")

    resolutions_path = None
    if resolutions:
        resolutions_path = os.path.join(save_dir, f'{group_name}_mismatch_resolutions.json')
        serializable = {str(k): v for k, v in resolutions.items()}
        with open(resolutions_path, 'w') as f:
            json.dump(serializable, f, indent=2)
        print(f"Saved mismatch resolutions -> {resolutions_path}")

    return {'table': table_path, 'resolutions': resolutions_path}


def load_master_cell_table(save_dir, group_name):
    """
    Reload one tracking group's saved master table -- undoes
    save_master_cell_table's to_csv. Means Track A's later revisit doesn't
    need to re-run tracking from scratch just to get master_df back --
    exactly the gap this fix closes.

    Parameters
    ----------
    save_dir : str
    group_name : str

    Returns
    -------
    master_df : pandas.DataFrame
    """
    table_path = os.path.join(save_dir, f'{group_name}_master_table.csv')
    master_df = pd.read_csv(table_path)
    print(f"Loaded master table <- {table_path} ({len(master_df)} rows)")
    return master_df


## Function 2.8 — `flag_layer_mismatches`

Generalizes `MergeTrackedLayerSMI.py`'s pairwise `n_layer_changes` check to
however many `layer_<label>` columns are in the master table (2-3+, not
just 2) — identifies tracked cells whose Phase 1 layer label disagrees
across sessions in the group. These are exactly the cells Function 2.9
will let you resolve by hand.

Note: `nunique` ignores `None`/`NaN` by default, so a cell missing a layer
label in one session (shouldn't happen given Phase 1's exhaustive
assignment, but not impossible) could mask a real mismatch between its
*other* sessions. Not fixing that now since it shouldn't occur in practice
-- flagging it here so it's not forgotten if it ever comes up.

- **Input:** `master_df` (from `build_master_cell_table`).
- **Output:** `master_df` (same table + new boolean `layer_mismatch`
  column), `mismatched_rows` (just the flagged rows, for feeding into
  Function 2.9).

In [ ]:
def flag_layer_mismatches(master_df):
    """
    Identify tracked cells whose Phase 1 layer label differs across
    sessions in the group.

    Parameters
    ----------
    master_df : pandas.DataFrame
        From build_master_cell_table.

    Returns
    -------
    master_df : pandas.DataFrame
        Same table, with an added boolean 'layer_mismatch' column.
    mismatched_rows : pandas.DataFrame
        Just the rows where layer_mismatch is True, sorted by global_cell_id.
    """
    layer_cols = [c for c in master_df.columns if c.startswith('layer_')]

    master_df = master_df.copy()
    master_df['layer_mismatch'] = master_df[layer_cols].nunique(axis=1).gt(1)

    mismatched_rows = master_df[master_df['layer_mismatch']].sort_values('global_cell_id')

    n_mismatch = len(mismatched_rows)
    n_total = len(master_df)
    print(f"{n_mismatch}/{n_total} tracked cells have differing layer labels across sessions.")
    if n_mismatch > 0:
        print("\nMismatched cells:")
        print(mismatched_rows[['global_cell_id'] + layer_cols].to_string(index=False))

    return master_df, mismatched_rows


In [ ]:
# --- Try it on your real master table ---
master_df, mismatched_rows = flag_layer_mismatches(master_df)


## Postscript on the layer-mismatch investigation

The planned diagnostic below never got built -- instead, inspecting the
saved `{animal_id}_curve_boundary_consistency_averaged.png` directly
revealed 12/15 sessions needed a manual override anyway (real per-session
depth drift, not fixable by translation-only registration alone), which
led to building the vertical-nudge adjustment in Phase 1
(`1.LayerAssignment_Curve.py`'s `adjust_layer_curve_popup`).

That fix alone didn't fully explain the mismatch counts, though (94/133
even after nudging most sessions). The bigger culprit turned out to be a
separate bug **in this notebook**: `_load_layer_of_cell` was keying its
`{roi_idx: layer_name}` dict by the raw suite2p `stat.npy` index stored in
each Phase 1 `.h5` file, but `verified_matrix`/`registration_matrix`'s
`roi_idx` values (built from `load_session_for_tracking`'s
`footprints[i]`/`centroids[i]`) are POSITIONS within the iscell-filtered
list, not raw indices. Fixed directly in `_load_layer_of_cell` (see its
docstring). After both fixes: 3/119 mismatched cells -- consistent with
genuine borderline cases, not a systemic problem. Function 2.9 below
handles those.

## Function 2.9 -- `review_layer_mismatches_popup`

Interactive per-cell resolution for cells `flag_layer_mismatches` flagged.
Deliberately reuses `plot_cell_across_group`'s already-validated footprint-
crop visualization as-is (the same one `manual_review_matches` uses)
rather than recomputing depth/boundary geometry here -- Phase 1's saved
curve is in *different* coordinate conventions depending on whether a
session was `'propagated'` (anchor frame) vs `'overridden'` (that
session's own raw frame), and the exact shift Phase 1 used isn't even
persisted to disk, so re-deriving a boundary overlay in Phase 2 risks
introducing a new, subtle frame-convention bug for the sake of a nicety.
The layer labels shown are the already-computed, authoritative ones from
the master table -- only added as text context, never recalculated.

Controls: a number key (`'1'`-`'9'`) trusts that session's layer label as
the consensus, `'x'` excludes the cell from layer-stratified analyses,
`'b'`/Left goes back, `'q'` quits early (anything not yet decided stays
unresolved).

- **Input:** `tracking_result` (from `track_session_group`),
  `mismatched_rows` (from `flag_layer_mismatches`), `crop_size=40`,
  `microns_per_pixel`.
- **Output:** `resolutions` -- `{global_cell_id: {'action': 'trust_session'
  | 'excluded', 'trusted_label', 'consensus_layer'}}` for every cell you
  actually decided on before quitting/finishing.

In [ ]:
def review_layer_mismatches_popup(tracking_result, mismatched_rows, crop_size=40,
                                   microns_per_pixel=MICRONS_PER_PIXEL):
    """
    Interactive per-cell resolution for cells flag_layer_mismatches flagged
    as disagreeing across sessions. Reuses plot_cell_across_group's proven
    footprint-crop visualization so you can visually judge the cell's
    actual appearance/position across sessions, with each session's
    already-computed layer label (from the master table) shown as text.

    Controls
    --------
    '1'..'9'  -- trust that session's layer label as the consensus for this cell
    'x'       -- exclude this cell from layer-stratified analyses
    'b'/Left  -- back to the previous cell
    'q'       -- quit early (remaining cells stay unresolved)

    Parameters
    ----------
    tracking_result : dict
        From track_session_group.
    mismatched_rows : pandas.DataFrame
        From flag_layer_mismatches -- needs 'global_cell_id' and
        'layer_<label>' columns.
    crop_size : int
    microns_per_pixel : float

    Returns
    -------
    resolutions : dict
        {global_cell_id: {'action': 'trust_session' | 'excluded',
                           'trusted_label': str or None,
                           'consensus_layer': str or None}}
    """
    labels_order = tracking_result['labels_order']
    rows = mismatched_rows.reset_index(drop=True)
    total = len(rows)

    print(f"Reviewing {total} mismatched cells.")
    print("Controls: 'b'/Left = back, 'x' = exclude, 'q' = quit, "
          "or a number to trust that session:")
    for i, label in enumerate(labels_order, start=1):
        print(f"  {i}: {label}")
    print()

    resolutions = {}
    idx = 0

    while 0 <= idx < total:
        row = rows.iloc[idx]
        global_cell_id = int(row['global_cell_id'])

        fig = plot_cell_across_group(tracking_result, global_cell_id, crop_size=crop_size,
                                      microns_per_pixel=microns_per_pixel)

        layer_summary = "  |  ".join(f"{i}:{label}={row[f'layer_{label}']}"
                                      for i, label in enumerate(labels_order, start=1))
        status = resolutions.get(global_cell_id, {}).get('action', 'NOT REVIEWED')
        fig.suptitle(f"Mismatch review {idx+1}/{total} (cell {global_cell_id})  --  {status}\n"
                     f"{layer_summary}\n"
                     "Press a number to trust that session, 'x'=exclude, 'b'=back, 'q'=quit",
                     fontsize=11, fontweight='bold')

        action = {'value': None}
        valid_keys = {str(k): k - 1 for k in range(1, len(labels_order) + 1)}

        def on_key(event):
            if event.key in valid_keys:
                action['value'] = ('trust', valid_keys[event.key])
                fig.canvas.stop_event_loop()
            elif event.key == 'x':
                action['value'] = ('exclude', None)
                fig.canvas.stop_event_loop()
            elif event.key in ('b', 'left'):
                action['value'] = ('back', None)
                fig.canvas.stop_event_loop()
            elif event.key == 'q':
                action['value'] = ('quit', None)
                fig.canvas.stop_event_loop()

        fig.canvas.mpl_connect('key_press_event', on_key)
        plt.show(block=False)

        while action['value'] is None and plt.fignum_exists(fig.number):
            fig.canvas.start_event_loop(0.1)

        if plt.fignum_exists(fig.number):
            plt.close(fig)

        if action['value'] is None:
            print("\nWindow closed without a decision -- quitting review.")
            break

        kind, payload = action['value']

        if kind == 'trust':
            trusted_label = labels_order[payload]
            consensus_layer = row[f'layer_{trusted_label}']
            resolutions[global_cell_id] = {
                'action': 'trust_session', 'trusted_label': trusted_label,
                'consensus_layer': consensus_layer
            }
            print(f"  Cell {global_cell_id}: trusting {trusted_label} -> {consensus_layer}")
            idx += 1
        elif kind == 'exclude':
            resolutions[global_cell_id] = {'action': 'excluded', 'trusted_label': None,
                                            'consensus_layer': None}
            print(f"  Cell {global_cell_id}: excluded")
            idx += 1
        elif kind == 'back':
            idx = max(0, idx - 1)
        elif kind == 'quit':
            print("\nQuitting review.")
            break

    n_resolved = len(resolutions)
    n_unresolved = total - n_resolved
    print(f"\n  Resolved: {n_resolved}")
    print(f"  Unresolved: {n_unresolved} (treat as excluded until reviewed)")

    return resolutions

In [ ]:
# --- Try it on your real 3 mismatched cells ---
resolutions = review_layer_mismatches_popup(tracking_result, mismatched_rows)
resolutions

## Function 2.10 -- `track_and_build_table`

The one-call driver tying Functions 2.2-2.9 together for a single
comparison group -- this is what you actually run per group going forward,
instead of chaining the individual functions by hand each time.

Sequence: load sessions -> `track_session_group` -> `filter_and_flag_matches`
-> `manual_review_matches` (unless `skip_manual_review`) -> `save_tracking_group`
-> `build_master_cell_table` -> `flag_layer_mismatches` -> `review_layer_mismatches_popup`
(unless `skip_mismatch_review` or there's nothing flagged).

`skip_manual_review=True` gives you a quick/dry run using only the
auto-accepted cells (auto_reject and every manual_review cell excluded from
`verified_matrix`) -- useful for a fast look at a group before committing to
the real interactive review.

- **Input:** `session_catalog` (from `discover_animal_sessions`), `group_name`,
  `session_labels`, `reference_label`, plus the usual
  `filter_and_flag_matches`/`microns_per_pixel` knobs, `store_dir` (optional),
  `skip_manual_review`/`skip_mismatch_review` (both default `False`).
- **Output:** one dict bundling everything: `tracking_result`, `filter_report`,
  `cell_categories`, `verified_matrix`, `decisions`, `group_save_path`,
  `master_df`, `mismatched_rows`, `resolutions`.

Below that: the `TRACKING_GROUPS` config-loop convenience -- one entry per
comparison you care about (mirrors Phase 1's `ANIMALS_LAYER_CURVE` list
shape), looped to produce one result dict per group. This is the actual
entry point for processing your baseline/DCZ-pair/openloop comparisons.

In [ ]:
def track_and_build_table(session_catalog, group_name, session_labels, reference_label,
                           store_dir=None, max_footprint_pixels=500,
                           auto_accept_corr=0.7, auto_accept_dist=5.0,
                           microns_per_pixel=MICRONS_PER_PIXEL,
                           skip_manual_review=False, skip_mismatch_review=False):
    """
    One-call driver for a single comparison group: load -> track -> filter
    -> (manual review) -> save -> master table -> flag mismatches ->
    (mismatch review). Wraps Functions 2.2-2.9.

    Parameters
    ----------
    session_catalog : dict
        From discover_animal_sessions.
    group_name : str
    session_labels : list of str
        Must all be keys in session_catalog.
    reference_label : str
        Must be in session_labels.
    store_dir : str, optional
        Where to save the tracking group .h5 (defaults, inside
        save_tracking_group, to a TrackedGroups folder next to the
        reference session).
    max_footprint_pixels, auto_accept_corr, auto_accept_dist, microns_per_pixel :
        Passed through to filter_and_flag_matches.
    skip_manual_review : bool
        If True, skip the interactive manual_review_matches step --
        verified_matrix keeps only auto_accept cells (auto_reject and every
        manual_review cell excluded). Use for a quick/dry run.
    skip_mismatch_review : bool
        If True, skip review_layer_mismatches_popup -- mismatched cells stay
        flagged in master_df but unresolved.

    Returns
    -------
    result : dict with keys: tracking_result, filter_report, cell_categories,
        verified_matrix, decisions, group_save_path, master_df,
        mismatched_rows, resolutions, master_table_save_paths.
    """
    print("\n" + "=" * 90)
    print(f" TRACKING GROUP: {group_name}")
    print("=" * 90)

    sessions = {label: load_session_for_tracking(session_catalog[label]['plane0_path'])
                for label in session_labels}

    tracking_result = track_session_group(sessions, reference_label=reference_label,
                                           microns_per_pixel=microns_per_pixel)

    filtered_matrix, cell_categories, filter_report = filter_and_flag_matches(
        tracking_result, max_footprint_pixels=max_footprint_pixels,
        auto_accept_corr=auto_accept_corr, auto_accept_dist=auto_accept_dist,
        microns_per_pixel=microns_per_pixel
    )

    if skip_manual_review:
        verified_matrix = filtered_matrix.copy()
        for row, category in cell_categories.items():
            if category != 'auto_accept':
                verified_matrix[row, :] = -1
        decisions = {}
        print("skip_manual_review=True -- verified_matrix contains only auto_accept cells.")
    else:
        verified_matrix, decisions = manual_review_matches(
            tracking_result, cell_categories, microns_per_pixel=microns_per_pixel
        )

    plane0_paths = {label: session_catalog[label]['plane0_path'] for label in session_labels}
    group_save_path = save_tracking_group(
        tracking_result, verified_matrix, decisions, filter_report,
        plane0_paths, group_name, save_dir=store_dir
    )

    layer_curve_paths = {label: find_layer_curve_path(plane0_paths[label]) for label in session_labels}
    master_df = build_master_cell_table(tracking_result, verified_matrix, layer_curve_paths,
                                         microns_per_pixel=microns_per_pixel)

    master_df, mismatched_rows = flag_layer_mismatches(master_df)

    resolutions = {}
    if len(mismatched_rows) > 0 and not skip_mismatch_review:
        resolutions = review_layer_mismatches_popup(tracking_result, mismatched_rows,
                                                     microns_per_pixel=microns_per_pixel)
    elif len(mismatched_rows) > 0:
        print(f"skip_mismatch_review=True -- {len(mismatched_rows)} mismatched cells left unresolved.")

    # Persist master_df (+ mismatch resolutions) alongside save_tracking_group's
    # output -- see the fix note above Function save_master_cell_table for why
    # this wasn't happening before.
    master_table_save_paths = save_master_cell_table(
        master_df, group_name, save_dir=os.path.dirname(group_save_path), resolutions=resolutions
    )

    return {
        'tracking_result': tracking_result,
        'filter_report': filter_report,
        'cell_categories': cell_categories,
        'verified_matrix': verified_matrix,
        'decisions': decisions,
        'group_save_path': group_save_path,
        'master_df': master_df,
        'mismatched_rows': mismatched_rows,
        'resolutions': resolutions,
        'master_table_save_paths': master_table_save_paths,
    }


In [ ]:
# --- Real-use template: one entry per comparison group you want tracked ---
# Fill in real session labels (from session_catalog's printed output) for
# each comparison. reference_label must be one of that group's session_labels.

TRACKING_GROUPS = [
    {
        'group_name': 'Day1_vs_SalineDCZ1',
        'session_labels': ['Day1',
                            '260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE',
                            '260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ'],
        'reference_label': 'Day1',
    },
    # {
    #     'group_name': 'BaselineDay5_vs_DCZ2',
    #     'session_labels': ['Day5', 'DCZ2_SALINE', 'DCZ2_DCZ'],
    #     'reference_label': 'Day5',
    # },
    # {
    #     'group_name': 'OpenLoopActive',
    #     'session_labels': ['OpenLoopActive_SALINE', 'OpenLoopActive_DCZ'],
    #     'reference_label': 'OpenLoopActive_SALINE',
    # },
    # ... one entry per comparison you actually need.
]

group_results = {}
for cfg in TRACKING_GROUPS:
    group_results[cfg['group_name']] = track_and_build_table(
        session_catalog, cfg['group_name'], cfg['session_labels'], cfg['reference_label']
    )


## Interactive session picker (convenience, extends the Function 2.10 workflow)

Instead of hand-typing session labels into `SELECTED_LABELS`/`session_labels`,
pick them from a checkbox popup built from `session_catalog`. Built as two
small pieces plus a combined convenience:

1. **`select_sessions_popup`** -- checkbox list of every label in
   `session_catalog` (tagged with its `session_type`). Click boxes to
   toggle, click "Confirm selection" (or press Enter) when done. Closing
   without confirming just uses whatever was checked at that point, rather
   than erroring.
2. **`choose_reference_label_popup`** -- numbered-keypress picker for which
   of the *selected* sessions should be the tracking reference -- same
   numbered-choice convention `review_layer_mismatches_popup`'s "trust
   session" step already uses, for consistency rather than introducing a
   different widget style.
3. **`pick_tracking_group_interactively`** -- runs both in sequence and
   hands back exactly the `(session_labels, reference_label)` pair
   `track_and_build_table` needs, so you can skip writing a `TRACKING_GROUPS`
   entry by hand for a one-off group.

- **Input:** `session_catalog` (from `discover_animal_sessions`).
- **Output:** `session_labels` (list of str, the checked sessions),
  `reference_label` (str, the one you picked as reference).

In [ ]:
def select_sessions_popup(session_catalog, title='Select sessions for this tracking group'):
    """
    Checkbox popup listing every session in session_catalog. Click boxes to
    toggle, click 'Confirm selection' or press Enter when done.

    Parameters
    ----------
    session_catalog : dict
        From discover_animal_sessions.
    title : str

    Returns
    -------
    selected_labels : list of str
        Labels you checked, in session_catalog's original order.
    """
    labels = list(session_catalog.keys())
    n = len(labels)
    display_labels = [f"[{session_catalog[l]['session_type']:>8}] {l}" for l in labels]

    fig_height = max(4, 0.35 * n + 1.5)
    fig = plt.figure(figsize=(9, fig_height))
    try:
        fig.canvas.manager.set_window_title('SELECT SESSIONS -- check boxes, then Confirm (or Enter)')
    except Exception:
        pass

    fig.suptitle(title, fontsize=12, fontweight='bold')

    check_ax = fig.add_axes([0.05, 0.12, 0.9, 0.80])
    check = CheckButtons(check_ax, display_labels, [False] * n)

    confirm_ax = fig.add_axes([0.35, 0.02, 0.3, 0.06])
    confirm_button = Button(confirm_ax, 'Confirm selection')

    state = {'done': False, 'status': [False] * n}

    def on_confirm(event=None):
        state['status'] = list(check.get_status())
        state['done'] = True
        fig.canvas.stop_event_loop()

    confirm_button.on_clicked(on_confirm)

    def on_key(event):
        if event.key == 'enter':
            on_confirm()

    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.show(block=False)
    while not state['done'] and plt.fignum_exists(fig.number):
        fig.canvas.start_event_loop(0.1)

    if not state['done']:
        state['status'] = list(check.get_status())
        print("Window closed without pressing Confirm -- using current checkbox state anyway.")

    if plt.fignum_exists(fig.number):
        plt.close(fig)
        plt.pause(0.01)  # let Qt actually process the close/hide before we return

    selected_labels = [label for label, checked in zip(labels, state['status']) if checked]
    print(f"Selected {len(selected_labels)} sessions: {selected_labels}")

    return selected_labels


def choose_reference_label_popup(selected_labels):
    """
    Numbered-keypress picker for which of the selected sessions should be
    the tracking reference -- same convention review_layer_mismatches_popup
    uses for its 'trust session' choice.

    Parameters
    ----------
    selected_labels : list of str
        From select_sessions_popup (or any list you already have).

    Returns
    -------
    reference_label : str
    """
    n = len(selected_labels)
    if n == 0:
        raise ValueError("selected_labels is empty -- nothing to choose a reference from.")

    fig, ax = plt.subplots(figsize=(7, 1 + 0.4 * n))
    ax.axis('off')
    lines = [f"{i}: {label}" for i, label in enumerate(selected_labels, start=1)]
    ax.text(0.02, 0.5, "\n".join(lines), fontsize=12, va='center', family='monospace')
    fig.suptitle("Press the number of the session to use as reference", fontsize=12, fontweight='bold')
    try:
        fig.canvas.manager.set_window_title('CHOOSE REFERENCE SESSION')
    except Exception:
        pass

    state = {'choice': None}
    valid_keys = {str(k): k - 1 for k in range(1, n + 1)}

    def on_key(event):
        if event.key in valid_keys:
            state['choice'] = valid_keys[event.key]
            fig.canvas.stop_event_loop()

    fig.canvas.mpl_connect('key_press_event', on_key)
    plt.show(block=False)
    while state['choice'] is None and plt.fignum_exists(fig.number):
        fig.canvas.start_event_loop(0.1)
    if plt.fignum_exists(fig.number):
        plt.close(fig)
        plt.pause(0.01)  # let Qt actually process the close/hide before we return

    if state['choice'] is None:
        raise RuntimeError("No reference session chosen (window closed without pressing a number).")

    reference_label = selected_labels[state['choice']]
    print(f"Reference session: {reference_label}")
    return reference_label


def pick_tracking_group_interactively(session_catalog):
    """
    Runs select_sessions_popup then choose_reference_label_popup in
    sequence, handing back exactly what track_and_build_table needs.

    Parameters
    ----------
    session_catalog : dict
        From discover_animal_sessions.

    Returns
    -------
    session_labels : list of str
    reference_label : str
    """
    session_labels = select_sessions_popup(session_catalog)
    if len(session_labels) == 0:
        raise ValueError("No sessions were selected.")
    reference_label = choose_reference_label_popup(session_labels)
    return session_labels, reference_label


In [ ]:
# --- Try it: pick a group interactively instead of hand-typing labels ---
picked_labels, picked_reference = pick_tracking_group_interactively(session_catalog)
print("\npicked_labels:", picked_labels)
print("picked_reference:", picked_reference)

# Feed straight into the one-call driver:
# group_result = track_and_build_table(session_catalog, 'MyGroupName', picked_labels, picked_reference)
